In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import math
from collections import defaultdict
from tqdm import tqdm # For a nice progress bar


In [2]:

print("--- 1. RESTORING PROJECT STATE ---")

# A. Load Data
print("Loading datasets...")
train_df = pd.read_parquet('../data/100k/train_final_mini.parquet')
val_df = pd.read_parquet('../data/100k/val_final_mini.parquet')
all_items_df = pd.read_parquet('../data/100k/all_items_processed_100k.parquet')

# B. Load Artifacts
print("Loading embeddings and model...")
item_embeddings = np.load('../data/100k/item_embeddings_100k.npy')
model = tf.keras.models.load_model('../data/100k/two_tower_model_100k.keras')
user_emb_layer = model.get_layer('user_emb')

# C. Create Fast Lookup Maps
print("Building lookup maps...")
item_id_to_index = pd.Series(all_items_df.index, index=all_items_df['item_id']).to_dict()
index_to_item_id = {v: k for k, v in item_id_to_index.items()}
total_items = len(item_embeddings)

# ==========================================
# NEW: 2. PRE-COMPUTE DATA FOR ADVANCED METRICS
# ==========================================

# A. Build User History (The Mask)
# We need to know what users bought in the train set so we don't recommend it again
print("Building training interaction mask...")
user_history = defaultdict(list)
# itertuples is much faster than iterrows
for row in train_df.itertuples():
    if row.item_id in item_id_to_index:
        user_history[row.user_id].append(item_id_to_index[row.item_id])

# B. Calculate Item Popularity (For Novelty Metric)
print("Calculating item popularity...")
item_counts = train_df['item_id'].value_counts()
total_train_interactions = len(train_df)
# P(i) = count of item i / total interactions. If unseen, give a tiny probability.
min_prob = 1 / (total_train_interactions + 1)
item_popularity = {
    iid: count / total_train_interactions 
    for iid, count in item_counts.items()
}

# ==========================================
# 3. VECTORIZED EVALUATION LOOP
# ==========================================
TOP_K = 10
print(f"\n--- STARTING FULL GLOBAL EVALUATION (@K={TOP_K}) ---")
print(f"Testing on ALL {len(val_df)} users in validation set.")

metrics = {
    'hits': 0, 'ndcg': 0, 'mrr': 0, 
    'novelty_sum': 0, 'ild_sum': 0, 'count': 0
}
recommended_unique_indices = set()

# Using tqdm for a progress bar so you aren't left guessing!
for row in tqdm(val_df.itertuples(), total=len(val_df), desc="Evaluating Users"):
    uid = int(row.user_id)
    target_iid = int(row.item_id)
    
    if target_iid not in item_id_to_index: 
        continue # Skip target items not in our catalog
        
    target_idx = item_id_to_index[target_iid]
    
    # --- 1. Get User Vector ---
    uid_input = uid if uid < user_emb_layer.input_dim else 0
    user_vec = user_emb_layer(np.array([uid_input])).numpy().reshape(1, 32)
    
    # --- 2. Global Scoring (Vectorized) ---
    # Dot product of 1 user vs ALL 100k items at once!
    scores = np.dot(user_vec, item_embeddings.T).flatten()
    
    # --- 3. Apply Training Mask ---
    # Set the score of already-purchased items to negative infinity
    past_indices = user_history.get(uid, [])
    if past_indices:
        scores[past_indices] = -np.inf
        
    # --- 4. Fast Ranking ---
    # Instead of sorting all 100k items (slow), we calculate the exact rank 
    # by counting how many items have a higher score than our target item.
    target_score = scores[target_idx]
    rank = np.sum(scores > target_score) + 1
    
    # Find the actual Top-K items for Diversity/Novelty metrics
    # argpartition is a numpy trick that is much faster than argsort
    top_k_indices = np.argpartition(scores, -TOP_K)[-TOP_K:]
    # Sort just the top K to get them in exact order
    top_k_sorted = top_k_indices[np.argsort(scores[top_k_indices])[::-1]]
    
    # --- 5. Update Accuracy Metrics ---
    metrics['count'] += 1
    metrics['mrr'] += 1.0 / rank
    
    if rank <= TOP_K:
        metrics['hits'] += 1
        metrics['ndcg'] += 1.0 / math.log2(rank + 1)
        
    # --- 6. Update Beyond-Accuracy Metrics ---
    # A. Coverage
    recommended_unique_indices.update(top_k_sorted)
    
    # B. Novelty (-log2 of popularity)
    user_novelty = 0
    for idx in top_k_sorted:
        iid = index_to_item_id[idx]
        prob = item_popularity.get(iid, min_prob)
        user_novelty += -math.log2(prob)
    metrics['novelty_sum'] += (user_novelty / TOP_K)
    
    # C. Intra-List Diversity (ILD) using Cosine Similarity
    top_k_embs = item_embeddings[top_k_sorted]
    # Normalize embeddings to calculate cosine similarity via dot product
    norms = np.linalg.norm(top_k_embs, axis=1, keepdims=True)
    # Avoid divide by zero
    norms[norms == 0] = 1e-10 
    norm_embs = top_k_embs / norms
    sim_matrix = np.dot(norm_embs, norm_embs.T)
    # Average pairwise distance (1 - similarity), ignoring the diagonal (item vs itself)
    user_ild = 1.0 - (np.sum(sim_matrix) - TOP_K) / (TOP_K * (TOP_K - 1))
    metrics['ild_sum'] += user_ild

# --- FINAL RESULTS ---
n = metrics['count']
coverage = len(recommended_unique_indices) / total_items

print("\n" + "="*45)
print(f" 🏆 FINAL RESULTS ON VAL SET (N={n})")
print("="*45)
print(f"{'Recall/Hit Rate@'+str(TOP_K):<25} : {metrics['hits'] / n:.4f}")
print(f"{'NDCG@'+str(TOP_K):<25} : {metrics['ndcg'] / n:.4f}")
print(f"{'MRR (Mean Recip. Rank)':<25} : {metrics['mrr'] / n:.4f}")
print("-" * 45)
print(f"{'Catalog Coverage':<25} : {coverage * 100:.2f}%")
print(f"{'Avg Novelty (Higher=Rarer)':<25} : {metrics['novelty_sum'] / n:.4f}")
print(f"{'Intra-List Diversity':<25} : {metrics['ild_sum'] / n:.4f}")
print("="*45)

--- 1. RESTORING PROJECT STATE ---
Loading datasets...
Loading embeddings and model...
Building lookup maps...
Building training interaction mask...
Calculating item popularity...

--- STARTING FULL GLOBAL EVALUATION (@K=10) ---
Testing on ALL 10000 users in validation set.


Evaluating Users: 100%|█████████████████| 10000/10000 [00:54<00:00, 183.38it/s]


 🏆 FINAL RESULTS ON VAL SET (N=10000)
Recall/Hit Rate@10        : 0.0001
NDCG@10                   : 0.0001
MRR (Mean Recip. Rank)    : 0.0003
---------------------------------------------
Catalog Coverage          : 14.15%
Avg Novelty (Higher=Rarer) : 15.4914
Intra-List Diversity      : 0.1030


In [3]:
print("--- RUNNING POPULARITY BASELINE ---")

# 1. Find the top 10 most popular items in the TRAINING set
top_10_pop_items = train_df['item_id'].value_counts().head(10).index.tolist()

hits = 0
# 2. Evaluate on the VALIDATION set
for row in val_df.itertuples():
    target_iid = int(row.item_id)
    
    # Did the target item appear in the top 10 overall most popular?
    if target_iid in top_10_pop_items:
        hits += 1

pop_hit_rate = hits / len(val_df)
print(f"Popularity Baseline Hit Rate@10: {pop_hit_rate:.4f}")

--- RUNNING POPULARITY BASELINE ---
Popularity Baseline Hit Rate@10: 0.0099


In [4]:
print("--- MODEL DIAGNOSTIC ---")
# Pick one random user
sample_user_vec = user_emb_layer(np.array([10])).numpy().reshape(1, 32)

# Get scores for ALL 100k items for this one user
scores = np.dot(sample_user_vec, item_embeddings.T).flatten()

print(f"Max score: {np.max(scores):.4f}")
print(f"Min score: {np.min(scores):.4f}")
print(f"Mean score: {np.mean(scores):.4f}")
print(f"Standard Dev: {np.std(scores):.6f}")

--- MODEL DIAGNOSTIC ---
Max score: 0.0665
Min score: -0.0045
Mean score: 0.0285
Standard Dev: 0.006772


In [10]:
import os
# Fixes the OpenBLAS multi-threading warning
os.environ['OPENBLAS_NUM_THREADS'] = '1' 

import numpy as np
import pandas as pd
import implicit
import scipy.sparse as sparse
import json
import math
from collections import defaultdict
from tqdm import tqdm

print("=========================================")
print(" 🚀 PHASE 1: DATA SETUP & MATRIX BUILDING")
print("=========================================")

# 1. Load Data
print("Loading datasets...")
train_df = pd.read_parquet('../data/100k/train_final_mini.parquet')
val_df = pd.read_parquet('../data/100k/val_final_mini.parquet')
all_items_df = pd.read_parquet('../data/100k/all_items_processed_100k.parquet')

# 2. Load Mappings
print("Loading mapping dictionaries...")
def load_mapping(filename):
    with open(filename, 'r') as f:
        return json.load(f)

user_mapping = load_mapping('../data/user_mapping.json')
item_mapping = load_mapping('../data/item_mapping.json')

num_users = len(user_mapping)
num_items = len(item_mapping)
print(f"Total Universe -> Users: {num_users}, Items: {num_items}")

# 3. Build Sparse Matrix for ALS 
# 🚨 FIXED: Modern implicit requires a USER-ITEM matrix (Rows=Users, Cols=Items)
print("Building Sparse User-Item Matrix...")
row = train_df['user_id'].values
col = train_df['item_id'].values
data = np.ones(len(train_df)) 

user_item_data = sparse.csr_matrix((data, (row, col)), shape=(num_users, num_items))


print("\n=========================================")
print(" 🧠 PHASE 2: TRAINING ALS MODEL")
print("=========================================")

als_model = implicit.als.AlternatingLeastSquares(
    factors=32, 
    regularization=0.01, 
    iterations=15, 
    random_state=42
)

print("Fitting ALS model...")
als_model.fit(user_item_data)

als_user_embeddings = als_model.user_factors
als_item_embeddings = als_model.item_factors 

# =========================================================
# 🌟 THE CRITICAL FIX: ALIGN ALS MATRICES TO 100K CATALOG
# =========================================================
print("Slicing ALS item embeddings to match 100k catalog perfectly...")
# Extract just the raw IDs for our 100k items
valid_item_ids = all_items_df['item_id'].values

# Slice the giant ALS matrix to keep ONLY those 100k items, in the exact same order
als_item_embeddings_100k = als_item_embeddings[valid_item_ids]
print(f"Aligned Item Embeddings Shape: {als_item_embeddings_100k.shape}")


print("\n=========================================")
print(" 📊 PHASE 3: RIGOROUS EVALUATION (VAL SET)")
print("=========================================")

# 1. Prep Lookup Maps
item_id_to_index = pd.Series(all_items_df.index, index=all_items_df['item_id']).to_dict()
index_to_item_id = {v: k for k, v in item_id_to_index.items()}

# 2. Build Training Mask 
print("Building training interaction mask...")
user_history = defaultdict(list)
for r in train_df.itertuples():
    if r.item_id in item_id_to_index:
        user_history[r.user_id].append(item_id_to_index[r.item_id])

# 3. Calculate Item Popularity (For Novelty)
print("Calculating item popularity...")
item_counts = train_df['item_id'].value_counts()
total_train_interactions = len(train_df)
min_prob = 1 / (total_train_interactions + 1)
item_popularity = {iid: count / total_train_interactions for iid, count in item_counts.items()}

# 4. Evaluation Loop
TOP_K = 10
metrics = {'hits': 0, 'ndcg': 0, 'mrr': 0, 'novelty_sum': 0, 'ild_sum': 0, 'count': 0}
recommended_unique_indices = set()

print(f"\nStarting Global Ranking Evaluation (@K={TOP_K})...")
for r in tqdm(val_df.itertuples(), total=len(val_df), desc="Evaluating ALS"):
    uid = int(r.user_id)
    target_iid = int(r.item_id)
    
    if target_iid not in item_id_to_index: 
        continue
    target_idx = item_id_to_index[target_iid]
    
    if uid >= num_users:
        continue 
        
    user_vec = als_user_embeddings[uid].reshape(1, 32)
    
    # SCORE AGAINST THE SLICED 100K MATRIX ONLY
    scores = np.dot(user_vec, als_item_embeddings_100k.T).flatten()
    
    past_indices = user_history.get(uid, [])
    if past_indices:
        scores[past_indices] = -np.inf
        
    # +++ ADD THESE TWO LINES TO BREAK TIES +++
    # Adds microscopic random noise (e.g., 0.000000001) to every score
    noise = np.random.uniform(0, 1e-9, size=scores.shape)
    scores = scores + noise
    # +++++++++++++++++++++++++++++++++++++++++
        
    target_score = scores[target_idx]
    rank = np.sum(scores > target_score) + 1
    
    top_k_indices = np.argpartition(scores, -TOP_K)[-TOP_K:]
    top_k_sorted = top_k_indices[np.argsort(scores[top_k_indices])[::-1]]
    
    metrics['count'] += 1
    metrics['mrr'] += 1.0 / rank
    
    if rank <= TOP_K:
        metrics['hits'] += 1
        metrics['ndcg'] += 1.0 / math.log2(rank + 1)
        
    recommended_unique_indices.update(top_k_sorted)
    
    user_novelty = 0
    for idx in top_k_sorted:
        iid = index_to_item_id[idx]
        prob = item_popularity.get(iid, min_prob)
        user_novelty += -math.log2(prob)
    metrics['novelty_sum'] += (user_novelty / TOP_K)
    
    # Diversity against the sliced matrix
    top_k_embs = als_item_embeddings_100k[top_k_sorted]
    norms = np.linalg.norm(top_k_embs, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10 
    norm_embs = top_k_embs / norms
    sim_matrix = np.dot(norm_embs, norm_embs.T)
    user_ild = 1.0 - (np.sum(sim_matrix) - TOP_K) / (TOP_K * (TOP_K - 1))
    metrics['ild_sum'] += user_ild

n = metrics['count']
coverage = len(recommended_unique_indices) / len(all_items_df)

print("\n" + "="*45)
print(f" 🏆 ALS BASELINE FINAL RESULTS (N={n})")
print("="*45)
print(f"{'Recall/Hit Rate@'+str(TOP_K):<25} : {metrics['hits'] / n:.4f}")
print(f"{'NDCG@'+str(TOP_K):<25} : {metrics['ndcg'] / n:.4f}")
print(f"{'MRR (Mean Recip. Rank)':<25} : {metrics['mrr'] / n:.4f}")
print("-" * 45)
print(f"{'Catalog Coverage':<25} : {coverage * 100:.2f}%")
print(f"{'Avg Novelty (Higher=Rarer)':<25} : {metrics['novelty_sum'] / n:.4f}")
print(f"{'Intra-List Diversity':<25} : {metrics['ild_sum'] / n:.4f}")
print("="*45)

 🚀 PHASE 1: DATA SETUP & MATRIX BUILDING
Loading datasets...
Loading mapping dictionaries...
Total Universe -> Users: 915325, Items: 235824
Building Sparse User-Item Matrix...

 🧠 PHASE 2: TRAINING ALS MODEL
Fitting ALS model...


  0%|          | 0/15 [00:00<?, ?it/s]

Slicing ALS item embeddings to match 100k catalog perfectly...
Aligned Item Embeddings Shape: (48629, 32)

 📊 PHASE 3: RIGOROUS EVALUATION (VAL SET)
Building training interaction mask...
Calculating item popularity...

Starting Global Ranking Evaluation (@K=10)...


Evaluating ALS: 100%|███████████████████| 10000/10000 [00:27<00:00, 360.08it/s]


 🏆 ALS BASELINE FINAL RESULTS (N=10000)
Recall/Hit Rate@10        : 0.0003
NDCG@10                   : 0.0002
MRR (Mean Recip. Rank)    : 0.0003
---------------------------------------------
Catalog Coverage          : 86.90%
Avg Novelty (Higher=Rarer) : 15.8604
Intra-List Diversity      : 0.9614
